**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Intro to Artificial Neural Networks

We build a neural network **from scratch in NumPy** — every forward pass, every gradient, every update written by hand — and train it until a spiral no line could separate falls to a few dozen lines of code. After this, [PyTorch](../../Intro_DL_4_Physics/intro_pytorch/intro_pytorch.ipynb) will feel like a convenience, not a mystery.

## 0. Introduction

An artificial neuron computes $\sigma(\mathbf{w}^T\mathbf{x} + b)$: a weighted vote followed by a nonlinear squeeze. One neuron draws a single line through the data. The story of this workshop: *stack* votes into layers and the network bends that line into any boundary you need.

## 1. Pre-requisites

- [Intro to Python](../../Intro_Func_Prog/Intro_Python/Intro_Python.ipynb) — NumPy fluency.
- The chain rule from calculus — backpropagation *is* the chain rule, organized.
- [Adaptive Filtering: APA](../../Intro_Time_Series/Intro_AdFilt_APA.ipynb) is a helpful cousin: LMS is literally training a single linear neuron.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(1)

---
### 🕐 Session 1 of 3 — *From Neuron to Network* (~35 min)
**Goal:** understand the perceptron, why nonlinearity is essential, and why depth buys expressiveness.
**Feeds into:** Session 2 (backpropagation).

---

## 2. Theory: Neurons, Layers, Nonlinearity

### 2.1. The Dataset That Defeats a Line

Two interleaved spirals. No single weighted vote — no *line* — can separate them.

In [2]:
def spirals(n_per_class=300, noise=0.25):
    t = np.linspace(0.5, 3 * np.pi, n_per_class)
    X, y = [], []
    for cls, phase in [(0, 0.0), (1, np.pi)]:
        x1 = t * np.cos(t + phase) + noise * rng.standard_normal(n_per_class)
        x2 = t * np.sin(t + phase) + noise * rng.standard_normal(n_per_class)
        X.append(np.stack([x1, x2], axis=1)); y.append(np.full(n_per_class, cls))
    X = np.concatenate(X); y = np.concatenate(y)
    X = (X - X.mean(0)) / X.std(0)                # standardize
    return X, y

X, y = spirals()
plt.figure(figsize=(4.5, 4.5))
plt.scatter(*X[y == 0].T, s=8, label="class 0")
plt.scatter(*X[y == 1].T, s=8, label="class 1")
plt.legend(); plt.title("Try separating THIS with a line")
plt.axis("equal"); plt.tight_layout(); plt.show()

/tmp/ipykernel_1840136/294026007.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.axis("equal"); plt.tight_layout(); plt.show()


💡 **Intuition.** Why nonlinearity is non-negotiable: stacking *linear* layers collapses — a matrix times a matrix is just another matrix, so a 100-layer linear network is one line in disguise. The activation function between layers breaks that collapse. With it, each hidden neuron contributes one *fold* of the input space; layers of folds crumple the plane until the spirals become linearly separable in the last layer.

### 2.2. Activations

We'll use **ReLU** ($\max(0, u)$) in hidden layers — cheap, and its gradient doesn't vanish for active units — and a **sigmoid** on the output to read the result as a probability.

In [3]:
u = np.linspace(-4, 4, 200)
relu = np.maximum(0, u)
sigmoid = 1 / (1 + np.exp(-u))

fig, axes = plt.subplots(1, 2, figsize=(8, 2.4))
axes[0].plot(u, relu); axes[0].set_title("ReLU: max(0, u)")
axes[1].plot(u, sigmoid); axes[1].set_title("sigmoid: 1/(1+e⁻ᵘ)")
for ax in axes: ax.grid(True)
plt.tight_layout(); plt.show()

/tmp/ipykernel_1840136/1137002607.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 2 of 3 — *Backpropagation* (~35 min)
**Goal:** derive the gradients as the chain rule on a computational graph — and implement them.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (training).

---

## 3. Theory: Backpropagation

💡 **Intuition.** Backprop is *blame assignment*. The loss says "prediction off by this much." Walking backward through the network, each operation answers one local question: *given how much my output was to blame, how much were my inputs and weights to blame?* — that answer is its local derivative. Multiplying local blames along the path is exactly the chain rule; "backprop" is just doing it once per node, back-to-front, instead of re-deriving a formula per weight.

### 3.1. The Two-Layer Network

Forward pass, batch $X$ ($n \times 2$):

$$Z_1 = X W_1 + \mathbf{b}_1 \quad A_1 = \mathrm{ReLU}(Z_1) \quad Z_2 = A_1 W_2 + \mathbf{b}_2 \quad \hat{y} = \sigma(Z_2)$$

Loss — binary cross-entropy: $\;L = -\frac{1}{n}\sum y\log\hat{y} + (1-y)\log(1-\hat{y})$

### 3.2. The Gradients

The famous simplification: for sigmoid + cross-entropy, the output blame collapses to $\delta_2 = \hat{y} - y$ (predicted minus true — the *error*, again!). Then

$$\nabla W_2 = \tfrac{1}{n} A_1^T \delta_2 \qquad \delta_1 = (\delta_2 W_2^T) \odot \mathbf{1}[Z_1 > 0] \qquad \nabla W_1 = \tfrac{1}{n} X^T \delta_1$$

The ReLU mask $\mathbf{1}[Z_1>0]$ says: neurons that were off take no blame.

In [4]:
def init(sizes):
    params = {}
    for i, (fan_in, fan_out) in enumerate(zip(sizes[:-1], sizes[1:]), 1):
        params[f"W{i}"] = rng.standard_normal((fan_in, fan_out)) * np.sqrt(2 / fan_in)  # He init
        params[f"b{i}"] = np.zeros(fan_out)
    return params

def forward(p, X):
    Z1 = X @ p["W1"] + p["b1"]
    A1 = np.maximum(0, Z1)
    Z2 = A1 @ p["W2"] + p["b2"]
    yhat = 1 / (1 + np.exp(-Z2.ravel()))
    return yhat, (X, Z1, A1)

def backward(p, cache, yhat, y):
    X, Z1, A1 = cache
    m = len(y)
    d2 = (yhat - y).reshape(-1, 1)                    # output blame
    grads = {"W2": A1.T @ d2 / m, "b2": d2.mean(0)}
    d1 = (d2 @ p["W2"].T) * (Z1 > 0)                  # ReLU mask
    grads["W1"] = X.T @ d1 / m
    grads["b1"] = d1.mean(0)
    return grads

### 3.3. Trust, but Verify: the Gradient Check

The classic backprop bug is a silently wrong gradient. The antidote: compare against a finite difference $\frac{L(\theta + \epsilon) - L(\theta - \epsilon)}{2\epsilon}$ on a few random weights.

In [5]:
def loss_of(p, X, y):
    yhat, _ = forward(p, X)
    eps = 1e-12
    return -np.mean(y * np.log(yhat + eps) + (1 - y) * np.log(1 - yhat + eps))

p = init([2, 16, 1])
yhat, cache = forward(p, X)
grads = backward(p, cache, yhat, y)

eps = 1e-5
for name, idx in [("W1", (0, 3)), ("W2", (7, 0)), ("b1", (2,))]:
    p[name][idx] += eps;  lp = loss_of(p, X, y)
    p[name][idx] -= 2 * eps; lm = loss_of(p, X, y)
    p[name][idx] += eps
    numeric = (lp - lm) / (2 * eps)
    analytic = grads[name][idx]
    print(f"{name}{idx}: analytic {analytic:+.6f}  numeric {numeric:+.6f}")
    assert abs(numeric - analytic) < 1e-6

W1(0, 3): analytic -0.003470  numeric -0.003470
W2(7, 0): analytic -0.055097  numeric -0.055097
b1(2,): analytic +0.069507  numeric +0.069507


---
### 🕐 Session 3 of 3 — *Training the Network* (~40 min)
**Goal:** run the full training loop on the spiral, visualize the learned boundary, then meet PyTorch.
**Builds on:** Session 2. &nbsp; **Feeds into:** [Intro to PyTorch](../../Intro_DL_4_Physics/intro_pytorch/intro_pytorch.ipynb).

---

## 4. Application: Train on the Spiral

In [6]:
p = init([2, 32, 1])
lr, losses = 0.5, []

for epoch in range(3000):
    yhat, cache = forward(p, X)
    grads = backward(p, cache, yhat, y)
    for k in p:
        p[k] -= lr * grads[k]
    if epoch % 50 == 0:
        losses.append(loss_of(p, X, y))

acc = np.mean((forward(p, X)[0] > 0.5) == y)
print(f"final training accuracy: {acc:.1%}")

plt.figure(figsize=(7, 2.5))
plt.plot(np.arange(len(losses)) * 50, losses)
plt.xlabel("epoch"); plt.ylabel("cross-entropy loss"); plt.grid(True)
plt.title("The four-beat loop: predict, grade, diagnose, nudge")
plt.tight_layout(); plt.show()

final training accuracy: 100.0%


/tmp/ipykernel_1840136/3456064341.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [7]:
# Visualize what the network learned: paint every point of the plane by its prediction
g = np.linspace(-2.5, 2.5, 300)
GX, GY = np.meshgrid(g, g)
grid_pred = forward(p, np.stack([GX.ravel(), GY.ravel()], axis=1))[0].reshape(GX.shape)

plt.figure(figsize=(5, 4.5))
plt.contourf(GX, GY, grid_pred, levels=30, cmap="RdBu", alpha=0.7)
plt.colorbar(label="P(class 1)")
plt.scatter(*X[y == 0].T, s=6, c="darkred")
plt.scatter(*X[y == 1].T, s=6, c="navy")
plt.title("32 hidden ReLUs folded the plane around the spiral")
plt.axis("equal"); plt.tight_layout(); plt.show()

/tmp/ipykernel_1840136/1526725041.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.axis("equal"); plt.tight_layout(); plt.show()


Experiments worth 5 minutes each (edit and re-run):

- Hidden width 4 vs 32 vs 256 — watch the boundary sharpen (and eventually overfit the noise).
- Remove the ReLU (`A1 = Z1`) — the boundary collapses to a line, *proving* Session 1's claim.
- Learning rate 5.0 — meet divergence in person.

## 5. Conclusion

Everything deep learning does was in this notebook: forward pass, loss, chain-rule blame assignment, gradient step. Frameworks add autograd (no hand-derived gradients), GPU tensors, and libraries of layers — conveniences on top of *this* loop.

---
## Where next

- [Intro to PyTorch](../../Intro_DL_4_Physics/intro_pytorch/intro_pytorch.ipynb) — the same network with autograd doing Session 2 for you.
- [Convolutional Neural Networks](../README.md#workshop-2--convolutional-neural-networks-available) — weight sharing turns layers into learned filter banks (bridging back to [DSP](../../Intro_DSP/README.md)).
- [Intro to Transformers](../../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb) — attention as data-dependent connectivity.